# 01A · Resumen base de general splits

Panorama inicial del dashboard de equipos para la temporada 2024-25 en temporada regular.


## Configuración
Breve puesta a punto: importamos librerías, definimos rutas del proyecto y preparamos carpetas de salida.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Configuración visual básica
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (12, 6)

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[4]
DATA_DIR = PROJECT_ROOT / '00_data'
FIGURES_DIR = PROJECT_ROOT / '00e_reports' / 'figures' / 'base_overview'
TABLES_DIR = PROJECT_ROOT / '00e_reports' / 'tables' / 'base_overview'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RAW_REFERENCE_PATH = (
    DATA_DIR
    / '00a_raw'
    / 'team_dashboard'
    / 'team_dashboard_by_general_splits'
    / '2024-25'
    / 'Regular Season'
    / 'team_dashboard_by_general_splits__1610612737__dataset_0.parquet'
)
INTERMEDIATE_PATH = (
    DATA_DIR
    / '00b_intermediate'
    / 'team_dashboard'
    / 'general_splits'
    / '2024-25'
    / 'Regular Season'
    / 'team_dashboard__general_splits.parquet'
)
DATASET_REFERENCE_ID = 0

print(f'Proyecto: {PROJECT_ROOT}')
print(f'Parquet intermedio: {INTERMEDIATE_PATH}')
print(f'Parquet raw de referencia: {RAW_REFERENCE_PATH}')
print(f'Dataset de referencia (source_dataset): {DATASET_REFERENCE_ID}')


## Carga de datos
Leemos el parquet consolidado y seleccionamos el subset con `source_dataset = 0` para tomarlo como referencia de la estructura original.


In [ ]:
general_df = pd.read_parquet(INTERMEDIATE_PATH)
raw_reference_df = pd.read_parquet(RAW_REFERENCE_PATH)

source_dataset_col = next(
    (col for col in general_df.columns if col.lower() == 'source_dataset'),
    None,
)
if source_dataset_col is None:
    raise KeyError('No se encontró la columna de referencia `source_dataset` en el parquet intermedio.')

dataset_reference_df = general_df[general_df[source_dataset_col] == DATASET_REFERENCE_ID].copy()

print("general_df shape:", general_df.shape)
print("dataset_reference_df shape:", dataset_reference_df.shape)


### Vista rápida del parquet intermedio
Primeras filas para identificar columnas y formato de las métricas.


In [ ]:
display(general_df.head())


### Referencia `source_dataset = 0`
Visualizamos el subset usado como referencia para contrastar que conserva los campos originales.


In [ ]:
display(dataset_reference_df.head())


### Parquet raw de referencia (dataset 0)
Comprobamos el archivo original para validar que la estructura base coincide con el subset seleccionado.



In [ ]:
display(raw_reference_df.head())


## Validación Win/Loss por equipo
Corroboramos que cada franquicia tenga exactamente dos filas (victoria y derrota).


In [ ]:
wl_counts = (
    general_df.groupby(['TEAM_ID', 'TEAM_NAME'])['GAME_RESULT']
    .agg(unique_result_count='nunique', resultados=lambda x: sorted(x.unique()))
)

equipos_incompletos = wl_counts[wl_counts['unique_result_count'] != 2]
display(wl_counts.head())

if equipos_incompletos.empty:
    print('✅ Todos los equipos tienen registros de victoria y derrota.')
else:
    print('⚠️ Equipos con registros incompletos:')
    display(equipos_incompletos)


## Tablas resumen de métricas
Separamos métricas clave para victorias, derrotas y diferencias (W − L).


In [ ]:
metricas = [
    'W_PCT',
    'NET_RATING',
    'OFF_RATING',
    'DEF_RATING',
    'PACE',
    'EFG_PCT',
    'TOV_PCT',
    'OREB_PCT',
    'FTR',
]
clave_equipo = ['TEAM_ID', 'TEAM_NAME']

win_df = (
    general_df[general_df['GAME_RESULT'] == 'W']
    .sort_values('TEAM_NAME')
    .set_index(clave_equipo)
)
loss_df = (
    general_df[general_df['GAME_RESULT'] == 'L']
    .sort_values('TEAM_NAME')
    .set_index(clave_equipo)
)

win_table = win_df[metricas].rename(columns=lambda c: f'{c}_WIN')
loss_table = loss_df[metricas].rename(columns=lambda c: f'{c}_LOSS')
diff_table = (win_df[metricas] - loss_df[metricas]).rename(columns=lambda c: f'{c}_DIFF')

if {'W', 'L'}.issubset(general_df.columns):
    wl_resumen = (
        general_df.groupby(clave_equipo)[['W', 'L']].max()
        .assign(W_MINUS_L=lambda df: df['W'] - df['L'])
    )
    diff_table = diff_table.join(wl_resumen['W_MINUS_L'])
else:
    diff_table = diff_table.assign(W_MINUS_L=pd.NA)

win_table_reset = win_table.reset_index()
loss_table_reset = loss_table.reset_index()
diff_table_reset = diff_table.reset_index()

display(win_table_reset.head())
display(loss_table_reset.head())
display(diff_table_reset.head())

win_table_reset.to_csv(TABLES_DIR / 'team_metrics_win.csv', index=False)
loss_table_reset.to_csv(TABLES_DIR / 'team_metrics_loss.csv', index=False)
diff_table_reset.to_csv(TABLES_DIR / 'team_metrics_diff.csv', index=False)

print('Tablas exportadas a', TABLES_DIR)


## Rankings clave
Generamos rankings por diferencial de NET_RATING, diferencial W − L y los Four Factors.


In [ ]:
rank_net = (
    diff_table_reset
    .sort_values('NET_RATING_DIFF', ascending=False)
    .assign(NET_RATING_RANK=lambda df: df['NET_RATING_DIFF'].rank(method='dense', ascending=False).astype(int))
)

rank_wl = (
    diff_table_reset
    .sort_values('W_MINUS_L', ascending=False)
    .assign(W_MINUS_L_RANK=lambda df: df['W_MINUS_L'].rank(method='dense', ascending=False).astype(int))
)

four_factor_cols = ['EFG_PCT_DIFF', 'TOV_PCT_DIFF', 'OREB_PCT_DIFF', 'FTR_DIFF']
four_factor_rank = diff_table_reset[clave_equipo + four_factor_cols].copy()
orden_rangos = {
    'EFG_PCT_DIFF': False,
    'TOV_PCT_DIFF': True,
    'OREB_PCT_DIFF': False,
    'FTR_DIFF': False,
}
for col, asc in orden_rangos.items():
    rank_col = f"{col.replace('_DIFF', '')}_RANK"
    four_factor_rank[rank_col] = four_factor_rank[col].rank(method='dense', ascending=asc).astype(int)

rank_cols = [c for c in four_factor_rank.columns if c.endswith('_RANK')]
four_factor_rank['RANK_PROMEDIO'] = four_factor_rank[rank_cols].mean(axis=1)
four_factor_rank = four_factor_rank.sort_values('RANK_PROMEDIO')

display(rank_net.head())
display(rank_wl.head())
display(four_factor_rank.head())

rank_net.to_csv(TABLES_DIR / 'ranking_net_rating.csv', index=False)
rank_wl.to_csv(TABLES_DIR / 'ranking_w_minus_l.csv', index=False)
four_factor_rank.to_csv(TABLES_DIR / 'ranking_four_factors.csv', index=False)

print('Rankings exportados a', TABLES_DIR)


## ΔNET_RATING por equipo
Visualizamos la diferencia de NET_RATING entre partidos ganados y perdidos.


In [ ]:
net_diff_plot = diff_table_reset.sort_values('NET_RATING_DIFF', ascending=False)
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(
    data=net_diff_plot,
    x='NET_RATING_DIFF',
    y='TEAM_NAME',
    palette=sns.diverging_palette(240, 10, as_cmap=True),
    ax=ax,
)
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Diferencial de NET_RATING (W − L)')
ax.set_xlabel('Δ NET_RATING')
ax.set_ylabel('Equipo')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'delta_net_rating.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print('Figura guardada en', FIGURES_DIR / 'delta_net_rating.png')


## Comparativa Four Factors W vs L
Promedios de los cuatro factores en victorias y derrotas para toda la liga.


In [ ]:
four_factor_metricas = ['EFG_PCT', 'TOV_PCT', 'OREB_PCT', 'FTR']
four_factor_media = (
    general_df.groupby('GAME_RESULT')[four_factor_metricas]
    .mean()
    .reset_index()
)
four_factor_melt = four_factor_media.melt(
    id_vars='GAME_RESULT', value_vars=four_factor_metricas,
    var_name='Métrica', value_name='Valor'
)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=four_factor_melt, x='Métrica', y='Valor', hue='GAME_RESULT', ax=ax)
ax.set_title('Comparativa de Four Factors por resultado')
ax.set_xlabel('Factor')
ax.set_ylabel('Valor medio')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'four_factors_win_loss.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print('Figura guardada en', FIGURES_DIR / 'four_factors_win_loss.png')


## PACE vs OFF_RATING
Relación entre ritmo y eficiencia ofensiva separada por victorias y derrotas.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=general_df,
    x='PACE',
    y='OFF_RATING',
    hue='GAME_RESULT',
    style='GAME_RESULT',
    ax=ax,
)
ax.set_title('PACE vs OFF_RATING por resultado')
ax.set_xlabel('PACE')
ax.set_ylabel('OFF_RATING')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'pace_vs_off_rating.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print('Figura guardada en', FIGURES_DIR / 'pace_vs_off_rating.png')


## Distribución de NET_RATING
Histograma para comparar la distribución de NET_RATING entre victorias y derrotas.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    data=general_df,
    x='NET_RATING',
    hue='GAME_RESULT',
    element='step',
    stat='density',
    common_norm=False,
    kde=True,
    ax=ax,
)
ax.set_title('Distribución de NET_RATING')
ax.set_xlabel('NET_RATING')
ax.set_ylabel('Densidad')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'net_rating_distribution.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print('Figura guardada en', FIGURES_DIR / 'net_rating_distribution.png')


## Correlación de métricas
Heatmap de correlaciones para entender relaciones entre indicadores avanzados.


In [ ]:
corr_cols = [
    'W_PCT',
    'NET_RATING',
    'OFF_RATING',
    'DEF_RATING',
    'PACE',
    'EFG_PCT',
    'TOV_PCT',
    'OREB_PCT',
    'FTR',
]
corr_matrix = general_df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    ax=ax,
)
ax.set_title('Correlación entre métricas clave')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'metric_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print('Figura guardada en', FIGURES_DIR / 'metric_correlation_heatmap.png')
